# Mission 3 · 환자 증상 인식 — 학습 노트북

119 신고 전사 텍스트에서 9개 증상(고열, 구토, 두통, 복통, 어지러움, 열상, 오심, 전신쇠약, 호흡곤란)을 멀티라벨로 예측한다.

**최종 모델**: 사전학습 인코더 6개를 파인튜닝한 앙상블, Validation macro F1 **0.658** (기준표 0.64)

| 단계 | 내용 |
|---|---|
| 1. 데이터 | Training 라벨 JSON의 전사만 사용 (서울 구급 29,200통화) |
| 2. 학습 | 인코더 + 9개 sigmoid, 서로 다른 설정 6개 |
| 3. 임계값 | Training 내부 dev 10%에서만 증상별로 선택 |
| 4. 번들 | 6개 모델 + 토크나이저 + 임계값 → `mission3_ensemble.pt` 하나 |
| 5. 채점 | `inference.py`로 Validation 1회 채점 |

**규정 준수**: Validation은 학습·임계값 선택에 쓰지 않고 5단계 채점에만 쓴다. 입력은 전사 텍스트뿐이며, 전처리는 라벨과 무관하게 모든 통화에 같다.

In [1]:
import json, subprocess, sys
from collections import Counter
from pathlib import Path

import numpy as np
import torch

SUBMISSION = Path.cwd()                      # 저장소의 mission3/ 폴더
ROOT = SUBMISSION.parent                     # 저장소 루트
sys.path.insert(0, str(SUBMISSION))
from mission3_model import (LABELS, build_bundle, ensemble_probabilities, load_calls, macro_f1,
                            split_fit_dev, train_member, tune_thresholds)

TRAIN_LABEL_DIR = ROOT / 'data/raw/Training/2.라벨링데이터/TL_서울_구급'
VAL_AUDIO_DIR = ROOT / 'data/raw/Validation/1.원천데이터/VS_서울_구급'
VAL_LABEL_DIR = ROOT / 'data/raw/Validation/2.라벨링데이터/VL_서울_구급'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

/home/ldw2003/2026-data-creator-camp/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda NVIDIA A100 80GB PCIe


## 1. 데이터

라벨 JSON의 `utterances`를 시간 순서로 이어 붙이고 발화마다 `[대원]`/`[신고자]`를 붙인다. 정답은 `symptom` 중 9개 대상 증상만 남긴다(대상 외 증상은 제거하고 샘플은 유지).

In [2]:
train_calls = load_calls(TRAIN_LABEL_DIR, with_labels=True)
y = np.asarray([call['y'] for call in train_calls])
print(f'Training 통화 {len(train_calls):,}건')
print('통화당 증상 수:', dict(sorted(Counter(y.sum(1).tolist()).items())))
print()
for label, count in zip(LABELS, y.sum(0)):
    print(f'  {label:<6} {count:>5,}건 ({count / len(y):.1%})')
print()
print('통화당 전사 길이(글자) 중앙값:', int(np.median([len(call['text']) for call in train_calls])))

Training 통화 29,200건
통화당 증상 수: {1: 19560, 2: 7137, 3: 2161, 4: 325, 5: 17}

  고열     5,186건 (17.8%)
  구토     4,463건 (15.3%)
  두통     2,905건 (9.9%)
  복통     6,771건 (23.2%)
  어지러움   6,334건 (21.7%)
  열상     3,465건 (11.9%)
  오심     3,341건 (11.4%)
  전신쇠약   5,310건 (18.2%)
  호흡곤란   3,927건 (13.4%)

통화당 전사 길이(글자) 중앙값: 583


## 2. 학습 설정

모든 구성원은 같은 dev 분할(`data_seed=42`)을 쓰고, 아래 한 가지씩만 다르다. 같은 모델을 시드만 바꾼 앙상블은 판정이 98%까지 같아 효과가 작았기 때문에, 아키텍처와 손실 함수를 섞어 오류 양상을 다르게 했다.

공통: 최대 512 토큰, batch 32, AdamW lr 2e-5, OneCycle(warmup 10%), 3 epoch 중 dev macro F1 최고 epoch 저장.

In [3]:
MEMBERS = {
    #  이름       사전학습 모델            seed  손실   pos_weight 지수
    's42':  dict(model_name='klue/roberta-large', seed=42, loss='bce', pos_weight_power=1.0),
    's43':  dict(model_name='klue/roberta-large', seed=43, loss='bce', pos_weight_power=1.0),
    's44':  dict(model_name='klue/roberta-large', seed=44, loss='bce', pos_weight_power=1.0),
    'xlmr': dict(model_name='xlm-roberta-large',  seed=42, loss='bce', pos_weight_power=1.0),
    'pw05': dict(model_name='klue/roberta-large', seed=42, loss='bce', pos_weight_power=0.5),
    'asl':  dict(model_name='klue/roberta-large', seed=42, loss='asl', pos_weight_power=1.0),
}
fit_calls, dev_calls = split_fit_dev(train_calls)
print(f'학습 {len(fit_calls):,}건 / dev {len(dev_calls):,}건 (Training 내부 분할)')

학습 26,280건 / dev 2,920건 (Training 내부 분할)


## 3. 학습

`RETRAIN=True`면 6개 모델을 이 노트북에서 다시 학습한다(A100 1장 기준 모델당 약 15분). 제출한 가중치는 같은 학습 코드(`mission3_model.train_member`와 동일한 루프)로 학습한 실행의 결과이며, 아래에 그 실행의 학습 로그를 그대로 싣는다.

In [4]:
RETRAIN = False
MEMBER_CKPT = {
    's42':  ROOT / 'checkpoints/mission3_roberta.pt',
    's43':  ROOT / 'checkpoints/mission3_roberta_s43.pt',
    's44':  ROOT / 'checkpoints/mission3_roberta_s44.pt',
    'xlmr': ROOT / 'checkpoints/mission3_xlmr.pt',
    'pw05': ROOT / 'checkpoints/mission3_roberta_pw05.pt',
    'asl':  ROOT / 'checkpoints/mission3_roberta_asl.pt',
}
LOG_FILES = {
    's42': 's42_klue-roberta-large.log', 's43': 's43_klue-roberta-large.log',
    's44': 's44_klue-roberta-large.log', 'xlmr': 'xlmr_xlm-roberta-large.log',
    'pw05': 'pw05_klue-roberta-large.log', 'asl': 'asl_klue-roberta-large.log',
}

if RETRAIN:
    retrained = SUBMISSION / 'member_ckpt'
    retrained.mkdir(exist_ok=True)
    for name, config in MEMBERS.items():
        print(f'===== {name} =====')
        MEMBER_CKPT[name] = retrained / f'{name}.pt'
        train_member(train_calls, output=MEMBER_CKPT[name], device=DEVICE, **config)
else:
    for name, log in LOG_FILES.items():
        print(f'===== {name}: {MEMBERS[name]} =====')
        for line in (SUBMISSION / 'logs' / log).read_text().splitlines():
            if line.startswith(('epoch', '{"epoch', 'saved')):
                print(line)
        print()

===== s42: {'model_name': 'klue/roberta-large', 'seed': 42, 'loss': 'bce', 'pos_weight_power': 1.0} =====
epoch 1 step 100/822 loss=1.1723
epoch 1 step 200/822 loss=1.0257
epoch 1 step 300/822 loss=0.9376
epoch 1 step 400/822 loss=0.8882
epoch 1 step 500/822 loss=0.8568
epoch 1 step 600/822 loss=0.8345
epoch 1 step 700/822 loss=0.8162
epoch 1 step 800/822 loss=0.8013
{"epoch": 1, "dev": {"macro_f1": 0.6416910004368321, "micro_f1": 0.6434782608695652, "exact_match": 0.4126712328767123}}
epoch 2 step 100/822 loss=0.6552
epoch 2 step 200/822 loss=0.6572
epoch 2 step 300/822 loss=0.6539
epoch 2 step 400/822 loss=0.6533
epoch 2 step 500/822 loss=0.6517
epoch 2 step 600/822 loss=0.6515
epoch 2 step 700/822 loss=0.6505
epoch 2 step 800/822 loss=0.6484
{"epoch": 2, "dev": {"macro_f1": 0.6493031333011738, "micro_f1": 0.6579930128900132, "exact_match": 0.43904109589041096}}
epoch 3 step 100/822 loss=0.5524
epoch 3 step 200/822 loss=0.5665
epoch 3 step 300/822 loss=0.5672
epoch 3 step 400/822 los

## 4. 앙상블 번들과 임계값

6개 모델의 확률을 평균한 뒤, 증상별 임계값을 **dev에서만** 고른다. 번들에는 토크나이저와 모델 설정 파일까지 넣어 채점 환경에서 인터넷 없이 동작하게 하고, 가중치는 float16으로 저장해 용량을 절반으로 줄인다. 임계값은 번들을 실제로 불러 계산한 확률로 고르므로 추론 경로와 정확히 같다.

In [5]:
bundle = build_bundle(MEMBER_CKPT)
dev_truth = np.asarray([call['y'] for call in dev_calls])
dev_prob = ensemble_probabilities(bundle, dev_calls, DEVICE)
bundle['thresholds'] = tune_thresholds(dev_truth, dev_prob)

dev_score = macro_f1(dev_truth, (dev_prob >= bundle['thresholds']).astype(int))
print(f'dev macro F1 (6개 앙상블): {dev_score:.4f}')
for label, threshold in zip(LABELS, bundle['thresholds']):
    print(f'  {label:<6} 임계값 {threshold:.2f}')

torch.save(bundle, SUBMISSION / 'mission3_ensemble.pt')
print(f"saved mission3_ensemble.pt ({(SUBMISSION / 'mission3_ensemble.pt').stat().st_size / 1e9:.2f} GB)")

  scored s42


  scored s43


  scored s44


  scored xlmr


  scored pw05


  scored asl


dev macro F1 (6개 앙상블): 0.6553
  고열     임계값 0.69
  구토     임계값 0.70
  두통     임계값 0.76
  복통     임계값 0.68
  어지러움   임계값 0.58
  열상     임계값 0.73
  오심     임계값 0.63
  전신쇠약   임계값 0.62
  호흡곤란   임계값 0.73


saved mission3_ensemble.pt (4.51 GB)


## 5. Validation 채점 (1회)

제출 명령과 똑같이 `inference.py`를 실행해 CSV를 만들고, 그 CSV를 Validation 정답과 비교한다. 정답은 채점할 때만 읽는다.

In [6]:
csv_path = SUBMISSION / 'results/mission3_validation.csv'
command = [sys.executable, 'inference.py', '--audio_dir', str(VAL_AUDIO_DIR), '--label_dir', str(VAL_LABEL_DIR),
           '--ckpt_path', 'mission3_ensemble.pt', '--output', str(csv_path)]
print(subprocess.run(command, capture_output=True, text=True, check=True).stdout)

import ast, csv
with csv_path.open(encoding='utf-8-sig') as stream:
    predicted = {row['label file name']: ast.literal_eval(row['symptom']) for row in csv.DictReader(stream)}
val_calls = load_calls(VAL_LABEL_DIR, with_labels=True)
truth = np.asarray([call['y'] for call in val_calls])
pred = np.asarray([[int(label in predicted[call['file']]) for label in LABELS] for call in val_calls])

from sklearn.metrics import f1_score
per_label = f1_score(truth, pred, average=None, zero_division=0)
metrics = {'macro_f1': macro_f1(truth, pred), 'micro_f1': float(f1_score(truth, pred, average='micro')),
           'exact_match': float((pred == truth).all(1).mean()), 'rows': len(predicted),
           'per_label': {label: float(score) for label, score in zip(LABELS, per_label)}}
(SUBMISSION / 'results/validation_metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2))
print(f"Validation macro F1 {metrics['macro_f1']:.4f} | micro {metrics['micro_f1']:.4f} | exact {metrics['exact_match']:.4f}")
for label, score in zip(LABELS, per_label):
    print(f'  {label:<6} F1 {score:.3f}  (정답 {int(truth[:, LABELS.index(label)].sum())}건)')

loaded 3,640 calls from /home/ldw2003/2026-data-creator-camp/data/raw/Validation/2.라벨링데이터/VL_서울_구급
  scored s42
  scored s43
  scored s44
  scored xlmr
  scored pw05
  scored asl
wrote /home/ldw2003/2026-data-creator-camp/mission3/results/mission3_validation.csv: 3,640 rows



Validation macro F1 0.6562 | micro 0.6678 | exact 0.4453
  고열     F1 0.694  (정답 678건)
  구토     F1 0.614  (정답 537건)
  두통     F1 0.541  (정답 374건)
  복통     F1 0.822  (정답 838건)
  어지러움   F1 0.667  (정답 774건)
  열상     F1 0.882  (정답 463건)
  오심     F1 0.403  (정답 429건)
  전신쇠약   F1 0.590  (정답 651건)
  호흡곤란   F1 0.694  (정답 492건)


## 6. 결과 해석

- 파인튜닝 한 가지로 프롬프트 방식 최고치(Gemma4-26B few-shot 0.599)보다 약 0.05 높아졌다. 앙상블은 여기에 약 0.01을 더했다.
- 열상·복통은 F1 0.8 이상이다. 오심은 0.40 수준인데, 오심 라벨 통화 대부분의 전사에 메스꺼움 표현이 없고, 반대로 표현이 있는데 라벨이 없는 통화도 있어 텍스트만으로는 한계가 있다.
- 서로 다른 모델(klue-roberta, xlm-roberta, Qwen3-4B)의 오답이 약 79% 겹쳤다. 남은 오류는 모델보다 데이터(라벨 기준)에서 온다.
- 시도했지만 효과가 없었던 것: 판정 매뉴얼 프롬프트, 증상 개수 기반 top-k, 학습 데이터 kNN 검색(RAG), train 100% 재학습(+0.002).